# 03 — Full training on Kaggle Notebooks (Loan Approval Predictor)

Why Kaggle instead of Colab: the Home Credit data mounts directly — no `kaggle.json`,
no 166 MB upload. Full guide in `KAGGLE.md`.

**Setup (one-time, in the Kaggle notebook sidebar):**
1. **+ Add Data → Competition → `Home Credit Default Risk`** (mounts at
   `/kaggle/input/home-credit-default-risk/`).
2. Notebook options (right panel / settings): **Internet ON** (needed for `pip install`
   and `git clone`), Accelerator **CPU** (SVC/XGBoost-hist here run on CPU; GPU buys little).
3. Recommended execution: **Save Version → Save & Run All (Commit)** so the multi-hour
   run executes headless (up to 9h) with versioned, downloadable outputs.

In [ ]:
# 0) Environment sanity + confirm the competition data is mounted.
import os, sys
print(sys.version)
DATA = "/kaggle/input/home-credit-default-risk/application_train.csv"
print("input dir:", os.listdir("/kaggle/input/home-credit-default-risk"))
assert os.path.exists(DATA), "Attach the competition data: + Add Data > Competition > Home Credit Default Risk"

In [ ]:
# 1) Clone the repo (needs Internet ON) and install pinned dependencies.
REPO_URL = "https://github.com/Shanmukh-Villuri/loan-approval-predictor.git"
!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
import sklearn, xgboost, imblearn, optuna
print("sklearn", sklearn.__version__, "| xgb", xgboost.__version__)
# If pinned installs fail (image Python != 3.11), see KAGGLE.md fallback.

In [ ]:
# 2) Fingerprint check (expect 307511x122, ~8.07% minority, 260 encoded features).
from src.data_loading import load_application_train, minority_rate
from src.preprocessing import add_engineered_features, infer_column_types, build_preprocessor, encoded_feature_count
df = load_application_train(DATA)
print("raw_shape=", df.shape, "minority_rate=", round(float(minority_rate(df["TARGET"])), 4))
feat = add_engineered_features(df.drop(columns=["TARGET", "SK_ID_CURR"]))
num, cat = infer_column_types(feat)
pre = build_preprocessor(num, cat).fit(feat)
print("numeric=", len(num), "categorical=", len(cat), "encoded=", encoded_feature_count(pre))

In [ ]:
# 3) Full training → outputs go to /kaggle/working (persisted in committed versions).
#    Hours-long: prefer Save & Run All (Commit) over interactive Run All.
#    Smoke test first if desired: !python -m src.train --quick --data DATA --out /kaggle/working/models_quick
!python -m src.train --data /kaggle/input/home-credit-default-risk/application_train.csv --out /kaggle/working/models

In [ ]:
# 4) Evaluate on held-out test + print the honest metrics.
!python -m src.evaluate --models-dir /kaggle/working/models --fig-dir /kaggle/working/reports/figures
!cat /kaggle/working/models/final_metrics.json

## 5) Bring results home

After the committed run finishes, open its **Output** tab and download:
`models/final_metrics.json`, `models/training_summary.json`, and every
`reports/figures/*.png`. Copy them into the laptop repo's `models/` and
`reports/figures/`, paste the ACTUAL numbers into the README metrics table,
and commit. Full checklist in `KAGGLE.md`.